# Estrategia Operacional de Pilas — SAG1 y SAG2
## Análisis Integral para Soporte a Decisiones

**División El Teniente — Codelco**  
**Período de análisis:** Enero–Junio 2026

---
### Skills aplicados
- `skill_molienda_sag` — contexto físico del proceso
- `skill_machine_learning_operacional` — modelos predictivos
- `skill_series_temporales_industriales` — análisis temporal
- `skill_data_scientist_senior` — metodología estadística
- `skill_product_owner_analitica_minera` — foco en decisión operacional

---
### Estructura del análisis

| Fase | Contenido |
|------|----------|
| 1 | Distribución real de pilas — percentiles y comportamiento histórico |
| 2 | Curva pila → TPH — LOWESS, no linealidades y puntos de quiebre |
| 3 | Zonas operacionales — Verde / Amarillo / Naranja / Rojo |
| 4 | Configuraciones históricas de molienda |
| 5 | Análisis de supervivencia operacional |
| 6 | Árbol de decisión operacional |
| 7 | Manual operacional — tablas y mapa de operación segura |
| 8 | Modelo dinámico ODE — ecuaciones diferenciales de inventario |

---
### Modelo físico (Fase 8)

$$\frac{dS_i}{dt} = Q_{T8,i}(t)\,[1-V(t)] - \sum_j Q_{SAG,ij}(t)$$

$$Q_{SAG,ij}(t) = A_{ij}(t) \cdot Q_{\max,ij} \cdot f(S_i) \cdot u_i(t)$$

$$f(S_i) = \frac{S_i}{S_i + K_S} \quad \text{(Michaelis-Menten)}$$

$$u_i(t) = \min\!\left(1,\; \frac{S_i - S_{\min}}{S_{\text{seg}} - S_{\min}}\right) \quad \text{(control operacional)}$$

In [ ]:
# Ejecutar análisis completo (Fases 1-7 y Fase 8)
import subprocess, sys
from pathlib import Path

BASE = Path('C:/Users/jorel038/OneDrive - Codelco/Documentos/AA_CIO_DET/07_Rendimientos')

print('Ejecutando Fases 1-7...')
r1 = subprocess.run([sys.executable, '-X', 'utf8', str(BASE / 'src/estrategia_pilas.py')],
                    capture_output=True, text=True, encoding='utf-8')
print(r1.stdout[-3000:] if len(r1.stdout) > 3000 else r1.stdout)
if r1.returncode != 0:
    print('STDERR:', r1.stderr[-1000:])

print('\nEjecutando Fase 8 (Modelo ODE)...')
r2 = subprocess.run([sys.executable, '-X', 'utf8', str(BASE / 'src/modelo_dinamico.py')],
                    capture_output=True, text=True, encoding='utf-8')
print(r2.stdout[-3000:] if len(r2.stdout) > 3000 else r2.stdout)
if r2.returncode != 0:
    print('STDERR:', r2.stderr[-1000:])

In [ ]:
# Mostrar figuras en el notebook
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

FIG = BASE / 'figures'

figuras = [
    ('F1_Distribucion_Pilas.png',        'Fase 1 — Distribución Real de Pilas'),
    ('F2_Curva_Pila_TPH.png',            'Fase 2 — Curva Pila → TPH (LOWESS)'),
    ('F3_Zonas_Operacionales.png',       'Fase 3 — Zonas Operacionales'),
    ('F4_Configuraciones_Molienda.png',  'Fase 4 — Configuraciones Históricas'),
    ('F5_Supervivencia_Operacional.png', 'Fase 5 — Supervivencia Operacional'),
    ('F6_Arbol_Decision.png',            'Fase 6 — Árbol de Decisión'),
    ('F7_Mapa_Operacion_Segura.png',     'Fase 7 — Mapa de Operación Segura'),
    ('F8_Validacion_Modelo.png',         'Fase 8 — Validación Modelo ODE'),
    ('F8_Michaelis_Menten.png',          'Fase 8 — Michaelis-Menten f(S)'),
    ('F8_Simulacion_Ventanas_T8.png',   'Fase 8 — Simulación Escenarios T8'),
    ('F8_Autonomia_Pilas.png',           'Fase 8 — Autonomía Operacional'),
    ('F8_Estrategia_Control.png',        'Fase 8 — Estrategia de Control'),
]

for fname, title in figuras:
    fpath = FIG / fname
    if not fpath.exists():
        print(f'No encontrado: {fname}')
        continue
    fig, ax = plt.subplots(figsize=(16, 9))
    img = mpimg.imread(str(fpath))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# Resumen ejecutivo de hallazgos
import json
with open(BASE / 'data/processed/estrategia_resultados.json') as f:
    res = json.load(f)

print('=' * 60)
print('RESUMEN EJECUTIVO — ESTRATEGIA OPERACIONAL DE PILAS')
print('=' * 60)
print()
for sag in ['SAG1','SAG2']:
    zonas = res['zonas'][sag]
    print(f'  {sag}:')
    print(f'    Verde    (operacion normal):   > {zonas["verde"][0]:.0f}%')
    print(f'    Amarillo (monitoreo activo):    {zonas["amarillo"][0]:.0f}% – {zonas["amarillo"][1]:.0f}%')
    print(f'    Naranja  (reducir carga):       {zonas["naranja"][0]:.0f}% – {zonas["naranja"][1]:.0f}%')
    print(f'    Rojo     (evaluar detención):   < {zonas["rojo"][1]:.0f}%')
    print()

print(f'  Tasa de descarga observada:')
print(f'    SAG1: {res["descarga_sag1_ph"]:.2f}%/h durante ventanas T8')
print(f'    SAG2: {res["descarga_sag2_ph"]:.2f}%/h durante ventanas T8')
print()
print('  Inventario minimo recomendado ANTES de ventana T8:')
print(f'    SAG1: > {max(res["zonas"]["SAG1"]["verde"][0], 60):.0f}% para ventanas > 8h')
print(f'    SAG2: > {res["zonas"]["SAG2"]["verde"][0]:.0f}% para ventanas > 8h')